# **Financial Data Retrieval**
---

This code efficiently retrieves returns, industry, volume, fundamental point-in-time (PiT) [*Market Cap, Book-Value, Price to Sales, Enterprise to EBITDA*] data for free using yfinance.

Currently this is implemented on S&P500 as it is the most readily available data. But the same method can be generalised.

---

In [ ]:
import requests
from bs4 import BeautifulSoup
from io import StringIO
import pandas as pd
import polars as pl
import yfinance as yf
from datetime import datetime
import pytz
import concurrent.futures
from tqdm import tqdm
import time
import numpy as np
import scipy as scipy

In [ ]:
# -----------------------------------------
# Step 1. Get S&P 500 Tickers
# -----------------------------------------
def get_sp500_tickers() -> list:
    """
    Scrapes the S&P 500 constituents from Wikipedia.
    Wraps the HTML in a StringIO object to avoid deprecation warnings.
    Returns a list of ticker symbols with dots replaced by hyphens.
    """
    url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
    resp = requests.get(url)
    df = pd.read_html(StringIO(resp.text))[0]
    tickers = df['Symbol'].str.strip().str.replace('.', '-', regex=False).tolist()
    return tickers

# -----------------------------------------
# Step 2. Download Historical Prices & Compute Returns
# -----------------------------------------
def download_price_data(tickers: list, start_date: str, end_date: str) -> pd.DataFrame:
    """
    Downloads historical price data for multiple tickers using yfinance in one bulk call.
    Handles invalid tickers gracefully and logs errors.
    """
    valid_tickers = []
    failed_tickers = []

    # Download data for all tickers
    data = None
    try:
        data = yf.download(tickers, start=start_date, end=end_date, progress=False)
        if isinstance(data.columns, pd.MultiIndex):
            # Use future_stack=True to avoid deprecation warnings
            data = data.stack(level=1, future_stack=True).rename_axis(['Date', 'symbol']).reset_index()
        else:
            data['symbol'] = tickers[0]
        valid_tickers = list(data['symbol'].unique())
    except Exception as e:
        print(f"Error downloading data: {e}")
        failed_tickers.extend(tickers)

    print(f"Valid tickers: {len(valid_tickers)}")
    print(f"Failed tickers: {len(failed_tickers)}")

    return data, valid_tickers, failed_tickers

def compute_asset_returns(price_df: pd.DataFrame) -> pd.DataFrame:
    """
    Computes daily asset returns based on the 'Close' price.
    Specifies fill_method=None to remove the pending future warning.
    Returns a DataFrame with columns: date, symbol, and asset_returns.
    """
    price_df = price_df.sort_values(['symbol', 'Date'])
    price_df['asset_returns'] = price_df.groupby('symbol')['Close'].pct_change(fill_method=None)
    ret_df = price_df[['Date', 'symbol', 'asset_returns']].rename(columns={'Date': 'date'})
    return ret_df

# -----------------------------------------
# Step 3. Fetch Fundamental Data from Yahoo Finance
# -----------------------------------------
def fetch_ticker_info(ticker: str, retries=3) -> dict:
    """
    Retrieves fundamental data for one ticker using yfinance.
    Includes retry logic with exponential backoff for rate-limiting errors.
    """
    time.sleep(0.5)  # Delay to not get limited
    try:
        info = yf.Ticker(ticker).info
        return {
            'symbol': ticker,
            'market_cap': info.get('marketCap'),
            'sector': info.get('sector'),
            'book_price': info.get('bookValue'),
            'sales_price': info.get('priceToSalesTrailing12Months'),
            'cf_price': info.get('enterpriseToEbitda')
        }
    except Exception as e:
        if retries > 0:
            time.sleep(2 ** (3 - retries))  # Exponential backoff
            return fetch_ticker_info(ticker, retries=retries-1)
        print(f"Error fetching info for {ticker}: {e}")
        return {
            'symbol': ticker,
            'market_cap': None,
            'sector': None,
            'book_price': None,
            'sales_price': None,
            'cf_price': None
        }

def get_fundamentals(tickers: list) -> pd.DataFrame:
    """
    Retrieves fundamental data for a list of tickers concurrently.
    Uses a ThreadPoolExecutor with fewer workers (to help avoid rate limiting)
    and a tqdm progress bar to track downloads.
    Returns a DataFrame with one row per ticker's fundamental information.
    """
    fundamentals = []

    # Use fewer workers to slow down concurrent requests.
    with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
        for info in tqdm(executor.map(fetch_ticker_info, tickers), total=len(tickers), desc="Fetching Fundamentals"):
            fundamentals.append(info)

    return pd.DataFrame(fundamentals)


# -----------------------------------------
# Step 4. Build Factor DataFrames
# -----------------------------------------
def build_factor_data(returns_df: pd.DataFrame, fundamentals_df: pd.DataFrame) -> tuple:
    """
    Efficiently builds factor DataFrames for the factor model.

    Parameters
    ----------
    returns_df: DataFrame with columns [date, symbol, asset_returns]
    fundamentals_df: DataFrame with current fundamental data

    Returns
    -------
    tuple of (mkt_cap_df, sector_df, style_df, value_df) where each is a DataFrame
    containing the factor data aligned with returns_df dates
    """

    symbols = fundamentals_df['symbol'].unique()
    mkt_cap_df = fundamentals_df[['symbol', 'market_cap']]
    fundamentals_df['sector'] = fundamentals_df['sector'].fillna("Unknown")
    sector_dummies = pd.get_dummies(fundamentals_df['sector'], prefix='sector')
    sector_df = pd.concat([fundamentals_df[['symbol']], sector_dummies], axis=1)

    # style factor (PIT only with Yfinance)
    style_df = fundamentals_df[['symbol', 'book_price', 'sales_price', 'cf_price']]
    value_df = style_df.copy()

    return mkt_cap_df, sector_df, style_df, value_df

In [ ]:
#-------------------#
# CORE DATA LOADING #
#-------------------#

period_start = "2015-01-01"
period_end = datetime.now(pytz.utc).strftime("%Y-%m-%d")

# Step 1: Retrieve tickers.
tickers = get_sp500_tickers()
print(f"Retrieved {len(tickers)} S&P 500 tickers.")

# Step 2: Download historical price data.
print("Downloading historical price data...")
price_df = download_price_data(tickers, period_start, period_end)
print(f"Downloaded price data with {price_df[0].shape[0]} records.")

Retrieved 503 S&P 500 tickers.
Valid tickers: 503
Failed tickers: 0
Downloaded price data with 1279129 records.


In [14]:
# Compute asset returns.
returns_df = compute_asset_returns(price_df[0])
print("Computed asset returns (sample):")
print(returns_df.head())

Computed asset returns (sample):
Price       date symbol  asset_returns
0     2015-01-02      A            NaN
503   2015-01-05      A      -0.018737
1006  2015-01-06      A      -0.015578
1509  2015-01-07      A       0.013272
2012  2015-01-08      A       0.029975


In [15]:
# Step 3: Fetch fundamental data with progress bar.
print("Fetching fundamental data for each ticker...")
fundamentals_df = get_fundamentals(tickers)
print("Sample fundamentals:")
print(fundamentals_df.head())

Fetching fundamental data for each ticker...


Fetching Fundamentals: 100%|██████████| 503/503 [01:07<00:00,  7.49it/s]

Sample fundamentals:
  symbol    market_cap       sector  book_price  sales_price  cf_price
0    MMM   80821518336  Industrials       8.524     2.476301    10.278
1    AOS    9460924416  Industrials      13.219     2.429741    12.266
2    ABT  226329706496   Healthcare      22.944     5.491174    19.405
3   ABBV  341058027520   Healthcare       3.413     6.141538    14.611
4    ACN  243205373952   Technology      45.242     3.747590    19.628


In [17]:
# Step 4: Build factor DataFrames
mkt_cap_df, sector_df, style_df, _ = build_factor_data(returns_df, fundamentals_df)

returns_pl = pl.DataFrame(returns_df)
mkt_cap_pl = pl.DataFrame(mkt_cap_df)
sector_pl = pl.DataFrame(sector_df)
style_pl = pl.DataFrame(style_df)

print(f"{returns_pl.tail()}\n{mkt_cap_pl.head()}\n {sector_pl.head()}\n{style_pl.head()}")

shape: (5, 3)
┌─────────────────────┬────────┬───────────────┐
│ date                ┆ symbol ┆ asset_returns │
│ ---                 ┆ ---    ┆ ---           │
│ datetime[ns]        ┆ str    ┆ f64           │
╞═════════════════════╪════════╪═══════════════╡
│ 2025-02-05 00:00:00 ┆ ZTS    ┆ 0.019914      │
│ 2025-02-06 00:00:00 ┆ ZTS    ┆ -0.008823     │
│ 2025-02-07 00:00:00 ┆ ZTS    ┆ -0.015449     │
│ 2025-02-10 00:00:00 ┆ ZTS    ┆ 0.0028        │
│ 2025-02-11 00:00:00 ┆ ZTS    ┆ 0.013844      │
└─────────────────────┴────────┴───────────────┘
shape: (5, 2)
┌────────┬──────────────┐
│ symbol ┆ market_cap   │
│ ---    ┆ ---          │
│ str    ┆ i64          │
╞════════╪══════════════╡
│ MMM    ┆ 80821518336  │
│ AOS    ┆ 9460924416   │
│ ABT    ┆ 226329706496 │
│ ABBV   ┆ 341058027520 │
│ ACN    ┆ 243205373952 │
└────────┴──────────────┘
 shape: (5, 12)
┌────────┬────────────┬────────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ symbol ┆ sector_Bas ┆ sector

**Save Data**

In [18]:
pl.DataFrame(price_df[0][["Date", "symbol", "Volume"]]).write_csv("volume_df.csv", separator=",")
returns_pl.write_csv("sp_500_ret.csv", separator=",")
mkt_cap_pl.write_csv("mkt_cap.csv", separator=",")
sector_pl.write_csv("sector_map.csv", separator=",")
style_pl.write_csv("style_factors.csv", separator=",")